# Discussion: predicting eight-day temperature range

The target is ΔTa = TX − TN, in °C. Because TX and TN are eight-day averages of daily maximum and minimum temperature, ΔTa represents the eight-day average daily temperature range, not the maximum minus minimum across the entire period (assuming matched daily coverage).

Predictors are LAI (leaf area per ground area, m²/m²), SM (volumetric soil moisture, m³/m³), HU (relative humidity, %) and TG (mean air temperature, °C).

We assumed dates label the start of eight-day windows and excluded windows crossing calendar-year boundaries. Models were fitted on 2010 and evaluated on 2011. Requiring the target and all four predictors gave 38, 42 and 34 training observations and 41, 43 and 35 test observations for the cells ordered by longitude. Identical complete-case rows were used for predictor comparisons within each cell. Window alignment remains an assumption supported by our comparison with supplied soil moisture.

The three locations were supplied with the challenge; the README does not specify their original selection method.


### What does the CSV contain, and do the time series show seasonality?

- The supplied dataset contains 276 observations, covering three grid cells during 2010–2011, with 92 eight-day observations per cell.
- It includes coordinates, dates, temperature variables, relative humidity and LAI. The supplied CSV also contained SM; we preserved it separately and calculated SM using the requested preprocessing.
- We calculated ΔTa from tx − tn and found no negative temperature ranges.


| Variable | What we observed |
|---|---|
| Mean temperature, TG | A clear annual cycle in all three cells, with higher values in summer. |
| LAI | A clear repeating seasonal cycle, with different magnitudes and timing between cells. |
| Humidity, HU | Strong seasonal variation in the middle cell; less regular variation in the other two. |
| Soil moisture, SM | Strong summer drying in the middle cell, a weaker seasonal pattern in the right cell, and relatively small variation in the left. |
| Temperature range, ΔTa | Different behaviour between cells; a summer increase is particularly apparent in the middle cell. |



The reasoning: the three locations have different environmental conditions, so we should not assume that the same predictor or regression relationship works equally well everywhere. The visible seasonal cycles also motivate checking whether predictor relationships simply reflect the time of year.

### why we prefer eight-day averaging over repeating or interpolating LAI to daily values.

| Approach | Advantage | Limitation |
|---|---|---|
| Average other variables to eight days | Matches LAI’s temporal resolution | Smooths short events and extremes; reduces sample size |
| Repeat LAI daily | Preserves daily meteorological variation | Creates abrupt LAI steps and adds no new vegetation observations |
| Interpolate LAI daily | Produces smooth daily estimates | Assumes how vegetation changes between observations; adds no measured daily information |


Averaging makes each row describe approximately the same time period. This suits our aim of explaining eight-day temperature-range variability.

Daily repetition or interpolation could be reasonable for a daily prediction task, but the extra rows would not represent additional independent LAI measurements. Interpolation using a later observation would also require care in a forecasting application.

The trade-off is that averaging can hide heatwaves, rainfall responses and other short-lived events. Our conclusions therefore apply to eight-day averages, rather than daily extremes.

I preferred eight-day averaging because it aligns the variables with LAI’s available resolution without assuming unobserved daily vegetation changes, while accepting the loss of short-term variability.

### Why did we apply these soil-moisture preprocessing steps?

1. Remove observations with uncertainty above 0.025 m³/m³.

This follows the specified threshold and excludes observations with higher reported uncertainty. It improves our confidence in the retained measurements, although it does not guarantee they are error-free.

2. Calculate an eight-day mean.

This aligns soil moisture with the temporal resolution of LAI and the meteorological variables. We average only the valid daily observations within each window.

3. Require at least five valid days per window.

Five days provide coverage of at least 62.5% of an eight-day period. This reduces the chance that one or two observations represent the entire window, while retaining more data than requiring all eight days.

The trade-off is that five available days may still miss an important wetting or drying event. Also, satellite gaps can depend on environmental conditions, so the retained sample may underrepresent conditions such as frozen soil or dense vegetation.

***Explanation***: We followed the specified uncertainty and coverage thresholds to balance measurement quality, temporal representativeness and data availability. A useful extension would test whether conclusions change with different thresholds.

### How do the three cells differ in hydrometeorological and vegetation conditions?

The time-series plots show:

| Cell (lon, lat) | Main characteristics |
|---|---|
| **−9.125, 52.125** | Highest soil moisture and persistently high humidity. Moderate LAI with a seasonal cycle. Relatively small temperature range. |
| **−4.125, 41.375** | Lowest soil moisture and LAI. Strong summer drying and lower summer humidity. Large seasonal temperature range. |
| **2.125, 45.875** | Intermediate soil moisture and generally high humidity. Highest peak LAI, with a pronounced growing season. Temperature range varies considerably. |

Why this matters: the cells cover contrasting environmental conditions. A predictor that varies strongly at one location may provide little information at another. For example, soil moisture changes relatively little in the first cell but shows pronounced drying in the second.

These are descriptions of the supplied data. We cannot identify vegetation type, irrigation or the exact causes of these differences from these variables alone.

### Which individual predictor explains ΔTa best? Does this differ between cells, and why?

Using the same complete-case 2010 observations for all four predictors:

| Cell (lon, lat) | Best individual predictor | Adjusted R² | Ordinary OLS p-value |
|---|---|---:|---:|
| −9.125, 52.125 | Mean temperature, TG | 0.136 | 0.0132 |
| −4.125, 41.375 | Relative humidity, HU | 0.890 | 5.14 × 10⁻²¹ |
| 2.125, 45.875 | Relative humidity, HU | 0.733 | 6.45 × 10⁻¹¹ |

Humidity is the strongest individual predictor in the second and third cells. Its negative slope means higher humidity is associated with a smaller temperature range.

In the first cell, TG ranks highest, but its fit is weak. “Best” here does not mean “good”: none of the individual predictors captures much of that cell’s variation.

The differences are consistent with the cells having different environmental conditions and predictor variability. Shared seasonal patterns can also strengthen these relationships, so these regressions do not isolate causal effects. Our later seasonal models address part of that issue.

These p-values use ordinary OLS assumptions. Our residual diagnostics concerned the selected multiple regressions, rather than every individual or seasonal model. They flag temporal dependence as a concern; temporal dependence and multiple comparisons warrant caution when interpreting the reported p-values.

### What do the sine and cosine variables describe, and should we include them?

They represent a smooth annual cycle:
\[
\text{seas\_sin}=\sin\left(\frac{2\pi t}{365}\right),
\qquad
\text{seas\_cos}=\cos\left(\frac{2\pi t}{365}\right)
\]
Here, \(t\) is the day of the year.

Why use both? Their fitted coefficients let the model adjust the cycle’s strength and timing. Using just one fixes where its peaks and troughs occur; combining them allows the peak to fall at different times of year.

Including them makes sense as a seasonal baseline. Temperature, vegetation and moisture can all vary seasonally. Adding each physical predictor to this baseline asks whether it provides information beyond a simple annual cycle.

There are two limits:
- One sine–cosine pair captures only a smooth annual pattern, not every seasonal process.
- An improvement beyond this baseline does not establish causation.
For our workflow: we included them to investigate seasonality, then excluded them from final model selection.

### Where do sine and cosine have explanatory power, and why?

| Cell (lon, lat) | Sine adjusted R² | Cosine adjusted R² | Both together |
|---|---:|---:|---:|
| −9.125, 52.125 | 0.402 | −0.021 | 0.385 |
| −4.125, 41.375 | 0.125 | 0.561 | 0.688 |
| 2.125, 45.875 | −0.031 | 0.225 | 0.201 |

- First cell: sine captures a substantial seasonal pattern; cosine alone contributes little.
- Second cell: cosine is stronger, and combining both gives the best seasonal fit. This indicates a pronounced annual pattern whose timing benefits from both terms.
- Third cell: cosine captures some variation, but the combined seasonal model has relatively limited explanatory power.
The differences reflect how closely each cell’s ΔTa follows a smooth annual cycle and when its highs and lows occur.

Why can adding cosine lower adjusted R²? It adds a coefficient. If the improvement in fit is too small, the complexity penalty outweighs it. Ordinary training R² cannot decrease when adding a predictor, but adjusted R² can.

### Which predictor adds the most explanatory power beyond seasonality, where, and why?

The values below are the increase in adjusted R² over the sine–cosine baseline:

| Cell (lon, lat) | + LAI | + SM | + HU | + TG | Largest gain |
|---|---:|---:|---:|---:|---|
| −9.125, 52.125 | −0.017 | **0.117** | 0.047 | 0.025 | Soil moisture |
| −4.125, 41.375 | 0.027 | 0.109 | **0.235** | 0.051 | Humidity |
| 2.125, 45.875 | 0.093 | 0.238 | **0.694** | 0.175 | Humidity |

Soil moisture adds the most in the first cell, even though its individual regression performed poorly. This suggests that accounting for the annual cycle reveals an association that the individual model did not capture.

Humidity adds the most in the second and third cells, especially the third. Its information therefore extends beyond the simple annual cycle.

Possible mechanisms include wetter surfaces allocating more available energy to evaporation, potentially limiting daytime warming, and atmospheric humidity and clouds influencing nighttime cooling. These are hypotheses consistent with physical processes, not causal effects demonstrated by these regressions.

LAI adds comparatively little. In the first cell, its negative gain means the small improvement in fit does not justify the additional coefficient.

### What is AIC, and how does it help select a lean model?

AIC balances how well a model fits the data against its complexity:
\[
AIC = 2k - 2\log(\hat L)
\]
Here, \(k\) is the number of estimated parameters and \(\hat L\) is the maximised likelihood. Lower AIC is preferred. Adding predictors must improve the fit enough to justify their extra parameters.

We compared models fitted to the same observations within each cell. AIC values from different cells are not directly comparable.

### Which models were retained, and are the thresholds appropriate?

We retained models with ΔAIC < 2 and adjusted R² > 0.36, then selected the one with the fewest predictors.

| Cell (lon, lat) | Retained models | Selected predictors | Training adjusted R² |
|---|---:|---|---:|
| −9.125, 52.125 | 1 | LAI + SM + HU + TG | 0.367 |
| −4.125, 41.375 | 5 | SM + HU | 0.898 |
| 2.125, 45.875 | 2 | SM + HU + TG | 0.908 |

Multiple retained models indicate model-selection uncertainty: several predictor combinations have similar support. Correlated predictors can partly substitute for one another.

ΔAIC < 2 is a useful screening rule, but adjusted R² > 0.36 is an arbitrary cutoff. In the first cell, it excludes simpler models just below 0.36 and forces selection of all four predictors. We followed the instructions, but would investigate threshold sensitivity and use validation within the training period to assess predictive performance.

### How good are the local predictions in 2011?

| Cell | Test RMSE | Training-mean baseline RMSE | Mean bias* |
|---|---:|---:|---:|
| −9.125, 52.125 | 2.035°C | 2.189°C | +1.393°C |
| −4.125, 41.375 | 2.971°C | 5.382°C | −2.458°C |
| 2.125, 45.875 | 1.763°C | 2.899°C | −0.542°C |

Bias = predicted − observed.

All models beat their baseline. The first cell improves only slightly; the second and third improve substantially. However, the second cell shows considerable underprediction despite its strong training fit.

### How does the pooled model compare?

The selected pooled model uses LAI + SM + HU, with one shared intercept and shared slopes.

| Cell | Local RMSE | Pooled RMSE | Better |
|---|---:|---:|---|
| −9.125, 52.125 | 2.035°C | 2.471°C | Local |
| −4.125, 41.375 | 2.971°C | 2.756°C | Pooled |
| 2.125, 45.875 | 1.763°C | 2.407°C | Local |

Pooling provides more training observations but assumes common relationships across locations. It helps the middle cell slightly and worsens the other two. A single shared model does not perform equally well everywhere.

### How can 2010 conditions affect prediction skill in 2011?

The models learn relationships from conditions represented in 2010. Their performance can decline if predictor distributions or relationships change in 2011.

In our complete-case samples:
- The first cell’s mean ΔTa decreased from 5.893 to 4.227°C, while predictor means remained similar. The model missed much of this decrease.
- The middle cell’s mean ΔTa increased from 11.920 to 14.802°C, contributing to underprediction.
- The third cell encountered humidity below its local training range, requiring extrapolation.
These observations are consistent with the errors, but do not establish their causes. One training year provides limited coverage of interannual variability.



### How would we predict ΔTa throughout Europe?

We would collect more years and geographically representative cells, covering different climates, elevations, vegetation and moisture conditions.

We would then:
- Harmonise spatial grids, temporal windows and quality filtering.
- Compare a shared model with regional models or models allowing location-dependent relationships.
- Validate on held-out years and held-out regions, assessing temporal and spatial transfer separately.
- Report performance by region and environmental condition, including extremes and missing-data coverage.
Our current experiment tests a new year at the same three locations. It does not demonstrate transfer to unseen European locations.

### What comes next, and what could confound the results?

Useful extensions include radiation, cloud cover, precipitation, wind, elevation and land cover. These may provide information missing from the current predictors.

We could also investigate physically motivated interactions, regularised regression and XGBoost, selecting features and tuning settings using validation within 2010. Because we have already inspected 2011 repeatedly, subsequent comparisons on that year should be described as exploratory.

Key limitations are:
- Correlated predictors make individual coefficients difficult to interpret.
- Remaining temporal dependence limits confidence in ordinary OLS p-values.
- Quality filtering and missing data may favour particular environmental conditions.
- Eight-day averaging smooths short events and extremes.
- Satellite surface soil moisture may not represent root-zone water availability.
- Two years and three cells provide limited evidence for wider generalisation.
Our overall finding: explanatory relationships differ strongly between cells. Humidity is particularly informative in two cells, soil moisture adds information beyond seasonality, and strong training fits do not guarantee reliable predictions in another year.